In [1]:
import os
import json
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

import faiss
import sentence_transformers

DEVICE = "cpu"
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.makedirs("./artifacts", exist_ok=True)
pd.set_option('display.max_colwidth', 1000)
pd.set_option('display.width', 1500)
pd.set_option('display.max_columns', None)

In [2]:
documents = json.load(open("./data/docs.json", encoding="utf-8"))["documents"]

In [3]:
df_documents = pd.DataFrame(documents)
df_documents.head()

,doc_id,title,author,year,text
0,classic_01,Преступление и наказание,Фёдор Достоевский,1866,"Социально-психологический и философский роман. Главный герой — бывший студент Родион Романович Раскольников, живущий в нищете в Петербурге, в своей каморке под самой крышей. Он создаёт теорию о делении людей на 'обычных' и 'право имеющих', которым дозволено проливать кровь ради великой цели. Чтобы проверить свою теорию и помочь семье, он решается на убийство старухи-процентщицы Алёны Ивановны. Однако сразу после преступления ему приходится убить и её случайно вернувшуюся сестру Лизавету. Действие происходит в душной атмосфере Петербурга, где Раскольников мечется между желанием сознаться и страхом разоблачения. Большую роль в сюжете играет следователь Порфирий Петрович, который догадывается о виновности Раскольникова и ведёт с ним тонкую психологическую дуэль. Важнейшей сюжетной линией является знакомство Раскольникова с Соней Мармеладовой, дочерью спившегося чиновника, вынужденной жить по 'жёлтому билету'. Именно Соня своим смирением и верой подводит героя к признанию и последующей..."
1,classic_02,Война и мир,Лев Толстой,1869,"Роман-эпопея, описывающая русское общество в эпоху войн против Наполеона (1805–1812 годы). В центре повествования — переплетение судеб нескольких дворянских семейств: Ростовых, Болконских, Курагиных и Безуховых. Наташа Ростова — живая, эмоциональная и музыкально одарённая девушка, чья первая любовь к князю Андрею Болконскому заканчивается трагически из-за его ранения при Бородине. Андрей Болконский — умный, скептичный и честолюбивый князь, который ищет смысл жизни сначала в военной славе (битва при Аустерлице), затем в уединённой деревенской жизни, а перед смертью приходит к христианскому всепрощению. Пьер Безухов — неуклюжий, богатый и вечно ищущий себя наследник графа, проходит путь от кутежей с Долоховым и неудачного брака с развратной Элен Курагиной до масонства и наблюдения за Бородинским сражением прямо на поле боя. В финале романа Пьер обретает счастье в браке с повзрослевшей Наташей Ростовой. Огромное место в книге занимают философские рассуждения автора о роли личности в и..."
2,classic_03,Мастер и Маргарита,Михаил Булгаков,1967,"Сатирический и мистический роман, действие которого происходит в Москве 1930-х годов. В город приезжает загадочный иностранец, профессор чёрной магии Воланд, который на самом деле является Сатаной. Его сопровождает свита: демон-шут Коровьев-Фагот в клетчатом пиджаке и треснувшем пенсне, кот Бегемот, любящий водку, шахматы и трамвайные приключения, мрачный убийца Азазелло и прекрасная ведьма Гелла. Воланд и его свита устраивают знаменитый сеанс чёрной магии в Театре Варьете, разоблачая жадность и лицемерие москвичей. Параллельно развивается трагическая история любви Мастера, написавшего роман о Понтии Пилате, и Маргариты Николаевны, которая ради спасения возлюбленного соглашается стать королевой на балу у Сатаны. В романе присутствует линия древнего Ершалаима, где прокуратор Иудеи Понтий Пилат мучается от головной боли и трусости, отправив на казнь бродячего философа Иешуа Га-Ноцри. Роман знаменит фразой: 'Рукописи не горят'."
3,classic_04,Евгений Онегин,Александр Пушкин,1833,"Роман в стихах, называемый 'энциклопедией русской жизни'. Главный герой — молодой дворянин Евгений Онегин, пресыщенный столичной жизнью, балами и театром. Получив в наследство имение дяди в деревне, он знакомится с юным поэтом-романтиком Владимиром Ленским и семьёй Лариных. Татьяна Ларина, задумчивая и мечтательная провинциальная девушка, влюбляется в Онегина с первого взгляда и пишет ему знаменитое письмо на французском языке, признаваясь в своих чувствах. Однако Онегин, пресыщенный и холодный, читает ей суровую отповедь в саду. Позже, на именинах Татьяны, Онегин от скуки начинает ухаживать за невестой Ленского, легкомысленной Ольгой, что приводит к дуэли, на которой Онегин убивает своего единственного друга. Спустя несколько лет Онегин встречает Татьяну в Петербурге уже блистательной замужней дамой, гене

In [4]:
df_documents.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   doc_id  15 non-null     object
 1   title   15 non-null     object
 2   author  15 non-null     object
 3   year    15 non-null     int64 
 4   text    15 non-null     object
dtypes: int64(1), object(4)
memory usage: 732.0+ bytes


In [5]:
class SentenceTransformersBackend:
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")


embedder = SentenceTransformersBackend(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device=DEVICE
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
def chunk_text(text: str, chunk_size: int = 30, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks

In [7]:
def build_chunks_dataframe(
    docs: List[Dict[str, str]],
    chunk_size: int = 10,
    overlap: int = 5,
) -> pd.DataFrame:
    rows = []

    for doc in docs:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk in enumerate(chunks):
            rows.append(
                {
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "chunk_id": chunk_id,
                    "chunk_text": chunk,
                    "n_words": len(chunk.split()),
                }
            )

    return pd.DataFrame(rows)


chunks_df = build_chunks_dataframe(documents, chunk_size=22, overlap=5)

print("Количество чанков:", len(chunks_df))
display(chunks_df.head(10))

Количество чанков: 126


,doc_id,title,chunk_id,chunk_text,n_words
0,classic_01,Преступление и наказание,0,"Социально-психологический и философский роман. Главный герой — бывший студент Родион Романович Раскольников, живущий в нищете в Петербурге, в своей каморке под самой",22
1,classic_01,Преступление и наказание,1,"в своей каморке под самой крышей. Он создаёт теорию о делении людей на 'обычных' и 'право имеющих', которым дозволено проливать кровь ради",22
2,classic_01,Преступление и наказание,2,"которым дозволено проливать кровь ради великой цели. Чтобы проверить свою теорию и помочь семье, он решается на убийство старухи-процентщицы Алёны Ивановны. Однако",22
3,classic_01,Преступление и наказание,3,убийство старухи-процентщицы Алёны Ивановны. Однако сразу после преступления ему приходится убить и её случайно вернувшуюся сестру Лизавету. Действие происходит в душной атмосфере,22
4,classic_01,Преступление и наказание,4,"Действие происходит в душной атмосфере Петербурга, где Раскольников мечется между желанием сознаться и страхом разоблачения. Большую роль в сюжете играет следователь Порфирий",22
5,classic_01,Преступление и наказание,5,"в сюжете играет следователь Порфирий Петрович, который догадывается о виновности Раскольникова и ведёт с ним тонкую психологическую дуэль. Важнейшей сюжетной линией является",22
6,classic_01,Преступление и наказание,6,"дуэль. Важнейшей сюжетной линией является знакомство Раскольникова с Соней Мармеладовой, дочерью спившегося чиновника, вынужденной жить по 'жёлтому билету'. Именно Соня своим смирением",22
7,classic_01,Преступление и наказание,7,билету'. Именно Соня своим смирением и верой подводит героя к признанию и последующей каторге в Сибири.,16
8,classic_02,Война и мир,0,"Роман-эпопея, описывающая русское общество в эпоху войн против Наполеона (1805–1812 годы). В центре повествования — переплетение судеб нескольких дворянских семейств: Ростовых, Болконских,",22
9,classic_02,Война и мир,1,"нескольких дворянских семейств: Ростовых, Болконских, Курагиных и Безуховых. Наташа Ростова — живая, эмоциональная и музыкально одарённая девушка, чья первая любовь к князю",22


In [8]:
chunk_texts = chunks_df["chunk_text"].tolist()
chunk_embeddings = embedder.fit_documents(chunk_texts)

chunk_embeddings

array([[-0.00184612,  0.03856223, -0.08905605, ...,  0.03756044,
        -0.101292  , -0.02865589],
       [-0.10327643,  0.14374919, -0.13501991, ..., -0.03628063,
         0.0067454 ,  0.00382258],
       [ 0.01168728,  0.07149066, -0.09144107, ..., -0.01080407,
         0.01781406,  0.02651026],
       ...,
       [ 0.029649  ,  0.061194  , -0.01555563, ...,  0.00442884,
        -0.04707246, -0.07105473],
       [-0.00255461,  0.05631842,  0.05722123, ..., -0.02626896,
        -0.01982858,  0.02756746],
       [ 0.10037815, -0.00363211,  0.01841219, ...,  0.00199931,
        -0.1018595 ,  0.04628762]], shape=(126, 384), dtype=float32)

In [9]:
class VectorSearchIndex:
    def __init__(self, dim: int) -> None:
        self.dim = dim
        self.backend_name = None
        self._faiss_index = None
        self._nn_index = None
        self._faiss_index = faiss.IndexFlatIP(dim)
        self.backend_name = "FAISS IndexFlatIP"

    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")
        self._faiss_index.add(vectors)

    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")
        scores, indices = self._faiss_index.search(query_vectors, top_k)
        return scores, indices


search_index = VectorSearchIndex(dim=chunk_embeddings.shape[1])
search_index.add(chunk_embeddings)

In [10]:
def search_similar_chunks(query: str, top_k: int = 5) -> pd.DataFrame:
    query_vectors = embedder.encode_queries([query])
    scores, indices = search_index.search(query_vectors, top_k=top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "doc_id": chunk_row["doc_id"],
                "title": chunk_row["title"],
                "chunk_id": int(chunk_row["chunk_id"]),
                "score": round(float(score), 4),
                "chunk_text": chunk_row["chunk_text"],
            }
        )

    return pd.DataFrame(rows)

In [11]:
questions = {
    # classic_01 - Преступление и наказание
    "Сколько шагов насчитал Раскольников от своей каморки до двери старухи-процентщицы и какой предмет он прикрепил к пальто, чтобы спрятать топор?": "classic_01",
    
    # classic_02 - Война и мир
    "Какое именно ранение получил Андрей Болконский под Аустерлицем и какие слова Наполеона он услышал, лежа на земле и глядя в небо?": "classic_02",
    
    # classic_03 - Мастер и Маргарита
    "Сколько тысяч червонцев выпало из кошелька у конферансье Жоржа Бенгальского после того, как Коровьев оторвал ему голову, и как звали кота, который пил водку и стрелял из маузера?": "classic_03",
    
    # classic_04 - Евгений Онегин
    "На каком языке Татьяна написала письмо Онегину и сколько дней он не появлялся в их доме после получения этого письма, прежде чем дать ей отповедь в саду?": "classic_04",
    
    # classic_05 - Мёртвые души
    "Сколько именно мёртвых душ удалось приобрести Чичикову у каждого из помещиков: Коробочки, Собакевича и Плюшкина, и кто продал их по самой низкой цене?": "classic_05",
    
    # classic_06 - Отцы и дети
    "Какой медицинский инструмент использовал Базаров при вскрытии тифозного больного, и сколько времени прошло с момента пореза до его смерти от заражения крови?": "classic_06",
    
    # classic_07 - Анна Каренина
    "Какую конкретную деталь в облике Анны заметил Вронский на вокзале при их первой встрече, и что произошло со смотрителем станции в тот же день, что стало зловещим предзнаменованием?": "classic_07",
    
    # classic_08 - Герой нашего времени
    "Какую песню пела Бэла за день до своей смерти и что именно попросила передать Печорину, когда поняла, что умирает?": "classic_08",
    
    # classic_09 - Собачье сердце
    "Какая кличка была у собаки до того, как её назвали Шарик, и сколько времени профессор Преображенский подбирал донора для пересадки гипофиза, прежде чем остановиться на Климе Чугункине?": "classic_09",
    
    # classic_10 - Тихий Дон
    "Сколько детей было у Григория Мелехова от Натальи и сколько от Аксиньи, и кто из них выжил после окончания Гражданской войны?": "classic_10",
    
    # classic_11 - 1984
    "Какой номер имела комната, которую снял Уинстон над лавкой мистера Чаррингтона для встреч с Джулией, и сколько времени (в минутах) показывали 'дыры забвения' в телеэкране, чтобы влюблённые могли побыть наедине?": "classic_11",
    
    # classic_12 - Горе от ума
    "Сколько лет не был в Москве Чацкий перед своим появлением в доме Фамусова, и какое конкретно звание и должность получил Молчалин, о чём и сообщил Чацкому в финале пьесы?": "classic_12",
}
for current_query in questions:
    display(Markdown(f"### Запрос: `{current_query}`"))
    display(search_similar_chunks(current_query, top_k=3))

### Запрос: `Сколько шагов насчитал Раскольников от своей каморки до двери старухи-процентщицы и какой предмет он прикрепил к пальто, чтобы спрятать топор?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,classic_01,Преступление и наказание,4,0.5248,"Действие происходит в душной атмосфере Петербурга, где Раскольников мечется между желанием сознаться и страхом разоблачения. Большую роль в сюжете играет следователь Порфирий"
1,2,classic_09,Собачье сердце,1,0.5133,"Филиппович Преображенский, занимающийся омоложением 'советских ответработников', решается на смелый эксперимент. Он пересаживает бездомному псу Шарику, подобранному в подворотне, семенные железы и гипофиз"
2,3,classic_01,Преступление и наказание,0,0.4988,"Социально-психологический и философский роман. Главный герой — бывший студент Родион Романович Раскольников, живущий в нищете в Петербурге, в своей каморке под самой"


### Запрос: `Какое именно ранение получил Андрей Болконский под Аустерлицем и какие слова Наполеона он услышал, лежа на земле и глядя в небо?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,classic_02,Война и мир,2,0.6167,"чья первая любовь к князю Андрею Болконскому заканчивается трагически из-за его ранения при Бородине. Андрей Болконский — умный, скептичный и честолюбивый князь,"
1,2,classic_02,Война и мир,6,0.5521,Курагиной до масонства и наблюдения за Бородинским сражением прямо на поле боя. В финале романа Пьер обретает счастье в браке с повзрослевшей
2,3,classic_08,Герой нашего времени,5,0.5453,"Пятигорске и убивает на дуэли своего старого знакомого Грушницкого. Повесть 'Фаталист' посвящена теме предопределения и судьбы. Несмотря на все свои дурные поступки,"


### Запрос: `Сколько тысяч червонцев выпало из кошелька у конферансье Жоржа Бенгальского после того, как Коровьев оторвал ему голову, и как звали кота, который пил водку и стрелял из маузера?`

...

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,classic_10,Тихий Дон,1,0.5444,"Гражданской войны. В центре повествования — жизнь Григория Мелехова, казака с хутора Татарский. Григорий мечется между двумя женщинами: законной, но нелюбимой женой"
1,2,classic_10,Тихий Дон,6,0.5215,"Наталью, дочь, отца, мать и Аксинью, убитую случайной пулей во время бегства с Кубани, — Григорий возвращается в родной хутор. Единственным живым"
2,3,classic_10,Тихий Дон,0,0.4938,"Роман-эпопея в четырёх томах о трагической судьбе донского казачества в переломную эпоху Первой мировой войны, революции и Гражданской войны. В центре повествования"


### Запрос: `Какой номер имела комната, которую снял Уинстон над лавкой мистера Чаррингтона для встреч с Джулией, и сколько времени (в минутах) показывали 'дыры забвения' в телеэкране, чтобы влюблённые могли побыть наедине?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,classic_11,1984,5,0.4616,"любовную связь с девушкой по имени Джулия, и они пытаются найти островок свободы в лесу или в комнате над лавкой антиквара. Их"
1,2,classic_11,1984,8,0.4161,"101, сломленный Уинстон сидит в кафе и смотрит на портрет Большого Брата с искренней любовью, поняв, что 'он любит Большого Брата'."
2,3,classic_11,1984,7,0.4122,"пыткам. Крысы — главный страх Уинстона — используются, чтобы сломать его окончательно. В финале, пройдя через комнату 101, сломленный Уинстон сидит в"


### Запрос: `Сколько лет не был в Москве Чацкий перед своим появлением в доме Фамусова, и какое конкретно звание и должность получил Молчалин, о чём и сообщил Чацкому в финале пьесы?`

,rank,doc_id,title,chunk_id,score,chunk_text
0,1,classic_12,Горе от ума,0,0.6689,"Комедия в стихах, разобранная на цитаты. Молодой дворянин Александр Андреевич Чацкий возвращается в Москву после трёхлетнего путешествия за границу. Он спешит в"
1,2,classic_07,Анна Каренина,1,0.6099,Москву помирить брата Стиву Облонского с его женой Долли. На вокзале она встречает блестящего молодого офицера графа Алексея Вронского. Их внезапная и
2,3,classic_09,Собачье сердце,0,0.5916,"Фантастическая сатирическая повесть. Действие происходит в Москве 1920-х годов в квартире на Пречистенке. Гениальный хирург, профессор Филипп Филиппович Преображенский, занимающийся омоложением 'советских"


In [12]:
results_top_2 = []

for query, expected in questions.items():
    answer_df = search_similar_chunks(query, top_k=2)
    retrieved_docs = answer_df["doc_id"].tolist()
    
    hit = int(expected in retrieved_docs)
    
    rank = None
    for i, doc_id in enumerate(retrieved_docs):
        if doc_id == expected:
            rank = i + 1
            break
    
    recall = hit  
    
    mrr = 1 / rank if rank else 0

    results_top_2.append({
        "query": query,
        "expected_source": expected,
        "retrieved_sources": retrieved_docs,
        "hit_at_k": hit,
        "recall_at_k": recall,
        "rank_of_first_relevant": rank,
        "mrr_at_k": round(mrr, 3)
    })

retrieval_eval = pd.DataFrame(results_top_2)
retrieval_eval.to_csv("./artifacts/retrieval_eval.csv")
hit_at_2 = retrieval_eval["hit_at_k"].sum()
print(f"hit@2: {hit_at_2}/12 = {hit_at_2/12*100:.1f}%")
mrr_sum = retrieval_eval["mrr_at_k"].sum()
print(f"MRR@2: {mrr_sum/12*100:.1f}%")
print(f"recall@2: {hit_at_2/12*100:.1f}%")
ranks = retrieval_eval["rank_of_first_relevant"].dropna()
print(f"\nРаспределение рангов:")
print(retrieval_eval["rank_of_first_relevant"].value_counts().sort_index())
retrieval_eval

hit@2: 10/12 = 83.3%
MRR@2: 83.3%
recall@2: 83.3%

Распределение рангов:
rank_of_first_relevant
1.0    10
Name: count, dtype: int64


,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,rank_of_first_relevant,mrr_at_k
0,"Сколько шагов насчитал Раскольников от своей каморки до двери старухи-процентщицы и какой предмет он прикрепил к пальто, чтобы спрятать топор?",classic_01,"[classic_01, classic_09]",1,1,1.0,1.0
1,"Какое именно ранение получил Андрей Болконский под Аустерлицем и какие слова Наполеона он услышал, лежа на земле и глядя в небо?",classic_02,"[classic_02, classic_02]",1,1,1.0,1.0
2,"Сколько тысяч червонцев выпало из кошелька у конферансье Жоржа Бенгальского после того, как Коровьев оторвал ему голову, и как звали кота, который пил водку и стрелял из маузера?",classic_03,"[classic_03, classic_08]",1,1,1.0,1.0
3,"На каком языке Татьяна написала письмо Онегину и сколько дней он не появлялся в их доме после получения этого письма, прежде чем дать ей отповедь в саду?",classic_04,"[classic_04, classic_04]",1,1,1.0,1.0
4,"Сколько именно мёртвых душ удалось приобрести Чичикову у каждого из помещиков: Коробочки, Собакевича и Плюшкина, и кто продал их по самой низкой цене?",classic_05,"[classic_05, classic_09]",1,1,1.0,1.0
5,"Какой медицинский инструмент использовал Базаров при вскрытии тифозного больного, и сколько времени прошло с момента пореза до его смерти от заражения крови?",classic_06,"[classic_09, classic_09]",0,0,NaN,0.0
6,"Какую конкретную деталь в облике Анны заметил Вронский на вокзале при их первой встрече, и что произошло со смотрителем станции в тот же день, что стало зловещим предзнаменованием?",classic_07,"[classic_07, classic_07]",1,1,1.0,1.0
7,"Какую песню пела Бэла за день до своей смерти и что именно попросила передать Печорину, когда поняла, что умирает?",classic_08,"[classic_06, classic_12]",0,0,NaN,0.0
8,"Какая кличка была у собаки до того, как её назвали Шарик, и сколько времени профессор Преображенский подбирал донора для пересадки гипофиза, прежде чем остановиться на Климе Чугункине?",classic_09,"[classic_09, classic_09]",1,1,1.0,1.0
9,"Сколько детей было у Григория Мелехова от Натальи и сколько от Аксиньи, и кто из них выжил после окончания Гражданской войны?",classic_10,"[classic_10, classic_10]",1,1,1.0,1.0


In [13]:
results_top_7 = []

for query, expected in questions.items():
    answer_df = search_similar_chunks(query, top_k=7)
    retrieved_docs = answer_df["doc_id"].tolist()
    
    hit = int(expected in retrieved_docs)
    
    rank = None
    for i, doc_id in enumerate(retrieved_docs):
        if doc_id == expected:
            rank = i + 1
            break
    
    recall = hit  
    
    mrr = 1 / rank if rank else 0

    results_top_7.append({
        "query": query,
        "expected_source": expected,
        "retrieved_sources": retrieved_docs,
        "hit_at_k": hit,
        "recall_at_k": recall,
        "rank_of_first_relevant": rank,
        "mrr_at_k": round(mrr, 3)
    })

retrieval_eval = pd.DataFrame(results_top_7)

hit_at_2 = retrieval_eval["hit_at_k"].sum()
print(f"hit@2: {hit_at_2}/12 = {hit_at_2/12*100:.1f}%")
mrr_sum = retrieval_eval["mrr_at_k"].sum()
print(f"MRR@2: {mrr_sum/12*100:.1f}%")
print(f"recall@2: {hit_at_2/12*100:.1f}%")
ranks = retrieval_eval["rank_of_first_relevant"].dropna()
print(f"\nРаспределение рангов:")
print(retrieval_eval["rank_of_first_relevant"].value_counts().sort_index())

retrieval_eval

hit@2: 12/12 = 100.0%
MRR@2: 87.5%
recall@2: 100.0%

Распределение рангов:
rank_of_first_relevant
1    10
3     1
6     1
Name: count, dtype: int64


,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,rank_of_first_relevant,mrr_at_k
0,"Сколько шагов насчитал Раскольников от своей каморки до двери старухи-процентщицы и какой предмет он прикрепил к пальто, чтобы спрятать топор?",classic_01,"[classic_01, classic_09, classic_01, classic_05, classic_01, classic_08, classic_05]",1,1,1,1.000
1,"Какое именно ранение получил Андрей Болконский под Аустерлицем и какие слова Наполеона он услышал, лежа на земле и глядя в небо?",classic_02,"[classic_02, classic_02, classic_08, classic_10, classic_06, classic_02, classic_10]",1,1,1,1.000
2,"Сколько тысяч червонцев выпало из кошелька у конферансье Жоржа Бенгальского после того, как Коровьев оторвал ему голову, и как звали кота, который пил водку и стрелял из маузера?",classic_03,"[classic_03, classic_08, classic_08, classic_05, classic_09, classic_03, classic_11]",1,1,1,1.000
3,"На каком языке Татьяна написала письмо Онегину и сколько дней он не появлялся в их доме после получения этого письма, прежде чем дать ей отповедь в саду?",classic_04,"[classic_04, classic_04, classic_12, classic_04, classic_05, classic_13, classic_03]",1,1,1,1.000
4,"Сколько именно мёртвых душ удалось приобрести Чичикову у каждого из помещиков: Коробочки, Собакевича и Плюшкина, и кто продал их по самой низкой цене?",classic_05,"[classic_05, classic_09, classic_05, classic_05, classic_09, classic_09, classic_06]",1,1,1,1.000
5,"Какой медицинский инструмент использовал Базаров при вскрытии тифозного больного, и сколько времени прошло с момента пореза до его смерти от заражения крови?",classic_06,"[classic_09, classic_09, classic_06, classic_09, classic_01, classic_06, classic_01]",1,1,3,0.333
6,"Какую конкретную деталь в облике Анны заметил Вронский на вокзале при их первой встрече, и что произошло со смотрителем станции в тот же день, что стало зловещим предзнаменованием?",classic_07,"[classic_07, classic_07, classic_12, classic_05, classic_01, classic_06, classic_07]",1,1,1,1.000
7,"Какую песню пела Бэла за день до своей смерти и что именно попросила передать Печорину, когда поняла, что умирает?",classic_08,"[classic_06, classic_12, classic_01, classic_05, classic_10, classic_08, classic_05]",1,1,6,0.167
8,"Какая кличка была у собаки до того, как её назвали Шарик, и сколько времени профессор Преображенский подбирал донора для пересадки гипофиза, прежде чем остановиться на Климе Чугункине?",classic_09,"[classic_09, classic_09, classic_09, classic_09, classic_09, classic_03, classic_09]",1,1,1,1.000
9,"Сколько детей было у Григория Мелехова от Натальи и сколько от Аксиньи, и кто из них выжил после окончания Гражданской войны?",classic_10,"[classic_10, classic_10, classic_10, classic_02, classic_02, classic_10, classic_10]",1,1,1,1.000


In [14]:
# новые доки в базу
new_documents = [
    {
        "doc_id": "new_01",
        "title": "Тёмные аллеи",
        "author": "Иван Бунин",
        "year": 1943,
        "text": "Сборник рассказов о любви, главный из которых — 'Тёмные аллеи' — повествует о встрече немолодого офицера Николая Алексеевича с бывшей крепостной Надеждой, которую он бросил 30 лет назад. Она стала хозяйкой постоялого двора, так и не простила его и не вышла замуж. Офицер осознаёт, что это была самая важная любовь в его жизни."
    },
    {
        "doc_id": "new_02",
        "title": "Доктор Живаго",
        "author": "Борис Пастернак",
        "year": 1957,
        "text": "Роман о судьбе врача и поэта Юрия Живаго на фоне революции и Гражданской войны. Ключевой сюжет: любовный треугольник между Юрием, его женой Тоней и Ларисой Антиповой. Юрий умирает от сердечного приступа в переполненном трамвае в 1929 году, не успев дописать свой цикл стихов."
    },
    {
        "doc_id": "new_03",
        "title": "Один день Ивана Денисовича",
        "author": "Александр Солженицын",
        "year": 1962,
        "text": "Повесть, описывающая один день из жизни заключённого Ивана Денисовича Шухова в сталинском лагере. Главный герой работает каменщиком на стройке, его главные заботы — не замёрзнуть, не попасть в карцер и раздобыть лишний кусок хлеба. Повесть принесла автору мировую известность."
    },
    {
        "doc_id": "new_04",
        "title": "Пиковая дама",
        "author": "Александр Пушкин",
        "year": 1834,
        "text": "Повесть о трёх заветных картах — тройке, семёрке и тузе. Инженер Германн узнаёт от графини тайну выигрыша, пугает её до смерти, затем является на похороны, где карта мигает ему. В конце концов, поставив все деньги на туза, вместо него вынимает даму пик и сходит с ума."
    }
]

update_questions = {**questions,
    "Кто написал 'Тёмные аллеи' и как зовут главного героя?": "new_01",
    "От чего умер Юрий Живаго в романе Пастернака?": "new_02",
    "Как зовут главного героя повести 'Один день Ивана Денисовича'?": "new_03",
    "Какие три карты были в 'Пиковой даме'?": "new_04",
}

In [15]:
before_sources = []

for query, expected in update_questions.items():
    answer_df = search_similar_chunks(query, top_k=7)
    retrieved_docs = answer_df["doc_id"].tolist()
    before_sources.append(retrieved_docs)

In [16]:
original_documents = documents.copy()
documents_updated = documents + new_documents

In [17]:
chunks_df_updated = build_chunks_dataframe(documents_updated, chunk_size=22, overlap=5)
print(f"Было чанков: {len(chunks_df)}")
print(f"Стало чанков: {len(chunks_df_updated)}")

Было чанков: 126
Стало чанков: 137


In [18]:
new_chunks = chunks_df_updated[chunks_df_updated["doc_id"].str.startswith("new_")]
new_chunks.head(5)

,doc_id,title,chunk_id,chunk_text,n_words
126,new_01,Тёмные аллеи,0,"Сборник рассказов о любви, главный из которых — 'Тёмные аллеи' — повествует о встрече немолодого офицера Николая Алексеевича с бывшей крепостной Надеждой,",22
127,new_01,Тёмные аллеи,1,"Алексеевича с бывшей крепостной Надеждой, которую он бросил 30 лет назад. Она стала хозяйкой постоялого двора, так и не простила его и",22
128,new_01,Тёмные аллеи,2,"и не простила его и не вышла замуж. Офицер осознаёт, что это была самая важная любовь в его жизни.",19
129,new_02,Доктор Живаго,0,"Роман о судьбе врача и поэта Юрия Живаго на фоне революции и Гражданской войны. Ключевой сюжет: любовный треугольник между Юрием, его женой",22
130,new_02,Доктор Живаго,1,"треугольник между Юрием, его женой Тоней и Ларисой Антиповой. Юрий умирает от сердечного приступа в переполненном трамвае в 1929 году, не успев",22


In [19]:
chunk_texts_updated = chunks_df_updated["chunk_text"].tolist()
chunk_embeddings_updated = embedder.fit_documents(chunk_texts_updated)

In [20]:
search_index_updated = VectorSearchIndex(dim=chunk_embeddings_updated.shape[1])
search_index_updated.add(chunk_embeddings_updated)
chunks_df = chunks_df_updated
search_index = search_index_updated

In [21]:
retrieval_after_update = []

for query, expected in update_questions.items():
    answer_df = search_similar_chunks(query, top_k=7)
    retrieved_docs = answer_df["doc_id"].tolist()
    
    hit = int(expected in retrieved_docs)
    
    rank = None
    for i, doc_id in enumerate(retrieved_docs):
        if doc_id == expected:
            rank = i + 1
            break
    
    recall = hit  
    
    mrr = 1 / rank if rank else 0

    retrieval_after_update.append({
        "query": query,
        "expected_source": expected,
        "retrieved_sources": retrieved_docs,
        "hit@k": hit,
        "recall@k": recall,
        "rank_of_first_relevant": rank,
        "mrr@k": round(mrr, 3)
    })

retrieval_after_update = pd.DataFrame(retrieval_after_update)
retrieval_after_update

,query,expected_source,retrieved_sources,hit@k,recall@k,rank_of_first_relevant,mrr@k
0,"Сколько шагов насчитал Раскольников от своей каморки до двери старухи-процентщицы и какой предмет он прикрепил к пальто, чтобы спрятать топор?",classic_01,"[new_03, classic_01, classic_09, classic_01, classic_05, new_03, classic_01]",1,1,2,0.500
1,"Какое именно ранение получил Андрей Болконский под Аустерлицем и какие слова Наполеона он услышал, лежа на земле и глядя в небо?",classic_02,"[classic_02, classic_02, new_03, classic_08, classic_10, classic_06, classic_02]",1,1,1,1.000
2,"Сколько тысяч червонцев выпало из кошелька у конферансье Жоржа Бенгальского после того, как Коровьев оторвал ему голову, и как звали кота, который пил водку и стрелял из маузера?",classic_03,"[classic_03, new_04, classic_08, classic_08, classic_05, classic_09, new_03]",1,1,1,1.000
3,"На каком языке Татьяна написала письмо Онегину и сколько дней он не появлялся в их доме после получения этого письма, прежде чем дать ей отповедь в саду?",classic_04,"[classic_04, classic_04, classic_12, classic_04, classic_05, new_02, classic_13]",1,1,1,1.000
4,"Сколько именно мёртвых душ удалось приобрести Чичикову у каждого из помещиков: Коробочки, Собакевича и Плюшкина, и кто продал их по самой низкой цене?",classic_05,"[classic_05, classic_09, classic_05, classic_05, classic_09, new_04, classic_09]",1,1,1,1.000
5,"Какой медицинский инструмент использовал Базаров при вскрытии тифозного больного, и сколько времени прошло с момента пореза до его смерти от заражения крови?",classic_06,"[classic_09, classic_09, classic_06, classic_09, classic_01, classic_06, classic_01]",1,1,3,0.333
6,"Какую конкретную деталь в облике Анны заметил Вронский на вокзале при их первой встрече, и что произошло со смотрителем станции в тот же день, что стало зловещим предзнаменованием?",classic_07,"[classic_07, classic_07, classic_12, classic_05, new_03, classic_01, classic_06]",1,1,1,1.000
7,"Какую песню пела Бэла за день до своей смерти и что именно попросила передать Печорину, когда поняла, что умирает?",classic_08,"[classic_06, new_04, classic_12, classic_01, classic_05, classic_10, classic_08]",1,1,7,0.143
8,"Какая кличка была у собаки до того, как её назвали Шарик, и сколько времени профессор Преображенский подбирал донора для пересадки гипофиза, прежде чем остановиться на Климе Чугункине?",classic_09,"[classic_09, classic_09, classic_09, classic_09, classic_09, classic_03, classic_09]",1,1,1,1.000
9,"Сколько детей было у Григория Мелехова от Натальи и сколько от Аксиньи, и кто из них выжил после окончания Гражданской войны?",classic_10,"[classic_10, classic_10, classic_10, classic_02, classic_02, classic_10, classic_10]",1,1,1,1.000


In [22]:
retrieval_before_after_update = pd.DataFrame()
retrieval_before_after_update["query"] = retrieval_after_update["query"]
retrieval_before_after_update["before_sources"] = before_sources
retrieval_before_after_update["after_sources"] = retrieval_after_update["retrieved_sources"]
retrieval_before_after_update["changed"] = retrieval_before_after_update["before_sources"] != retrieval_before_after_update["after_sources"]
retrieval_before_after_update.to_csv("./artifacts/retrieval_before_after_update.csv")
retrieval_before_after_update

,query,before_sources,after_sources,changed
0,"Сколько шагов насчитал Раскольников от своей каморки до двери старухи-процентщицы и какой предмет он прикрепил к пальто, чтобы спрятать топор?","[classic_01, classic_09, classic_01, classic_05, classic_01, classic_08, classic_05]","[new_03, classic_01, classic_09, classic_01, classic_05, new_03, classic_01]",True
1,"Какое именно ранение получил Андрей Болконский под Аустерлицем и какие слова Наполеона он услышал, лежа на земле и глядя в небо?","[classic_02, classic_02, classic_08, classic_10, classic_06, classic_02, classic_10]","[classic_02, classic_02, new_03, classic_08, classic_10, classic_06, classic_02]",True
2,"Сколько тысяч червонцев выпало из кошелька у конферансье Жоржа Бенгальского после того, как Коровьев оторвал ему голову, и как звали кота, который пил водку и стрелял из маузера?","[classic_03, classic_08, classic_08, classic_05, classic_09, classic_03, classic_11]","[classic_03, new_04, classic_08, classic_08, classic_05, classic_09, new_03]",True
3,"На каком языке Татьяна написала письмо Онегину и сколько дней он не появлялся в их доме после получения этого письма, прежде чем дать ей отповедь в саду?","[classic_04, classic_04, classic_12, classic_04, classic_05, classic_13, classic_03]","[classic_04, classic_04, classic_12, classic_04, classic_05, new_02, classic_13]",True
4,"Сколько именно мёртвых душ удалось приобрести Чичикову у каждого из помещиков: Коробочки, Собакевича и Плюшкина, и кто продал их по самой низкой цене?","[classic_05, classic_09, classic_05, classic_05, classic_09, classic_09, classic_06]","[classic_05, classic_09, classic_05, classic_05, classic_09, new_04, classic_09]",True
5,"Какой медицинский инструмент использовал Базаров при вскрытии тифозного больного, и сколько времени прошло с момента пореза до его смерти от заражения крови?","[classic_09, classic_09, classic_06, classic_09, classic_01, classic_06, classic_01]","[classic_09, classic_09, classic_06, classic_09, classic_01, classic_06, classic_01]",False
6,"Какую конкретную деталь в облике Анны заметил Вронский на вокзале при их первой встрече, и что произошло со смотрителем станции в тот же день, что стало зловещим предзнаменованием?","[classic_07, classic_07, classic_12, classic_05, classic_01, classic_06, classic_07]","[classic_07, classic_07, classic_12, classic_05, new_03, classic_01, classic_06]",True
7,"Какую песню пела Бэла за день до своей смерти и что именно попросила передать Печорину, когда поняла, что умирает?","[classic_06, classic_12, classic_01, classic_05, classic_10, classic_08, classic_05]","[classic_06, new_04, classic_12, classic_01, classic_05, classic_10, classic_08]",True
8,"Какая кличка была у собаки до того, как её назвали Шарик, и сколько времени профессор Преображенский подбирал донора для пересадки гипофиза, прежде чем остановиться на Климе Чугункине?","[classic_09, classic_09, classic_09, classic_09, classic_09, classic_03, classic_09]","[classic_09, classic_09, classic_09, classic_09, classic_09, classic_03, classic_09]",False
9,"Сколько детей было у Григория Мелехова от Натальи и сколько от Аксиньи, и кто из них выжил после окончания Гражданской войны?","[classic_10, classic_10, classic_10, classic_02, classic_02, classic_10, classic_10]","[classic_10, classic_10, classic_10, classic_02, classic_02, classic_10, classic_10]",False


In [23]:
class MiniRAG:

    def __init__(self, embedder, search_index, chunks_df):
        self.embedder = embedder
        self.search_index = search_index
        self.chunks_df = chunks_df
    
    def answer(self, query: str, top_k: int = 3) -> dict:
        """
        Основной метод RAG:
        1. Получить запрос пользователя
        2. Извлечь top-k релевантных фрагментов
        3. Собрать контекст
        4. Сформировать ответ
        5. Вернуть ответ + источники
        """
        query_vec = self.embedder.encode_queries([query])
        scores, indices = self.search_index.search(query_vec, top_k=top_k)
        
        context_parts = []
        sources = []
        retrieved_chunks = []
        
        for idx, score in zip(indices[0], scores[0]):
            chunk = self.chunks_df.iloc[int(idx)]
            context_parts.append(chunk["chunk_text"])
            sources.append(chunk["doc_id"])
            retrieved_chunks.append(chunk["chunk_text"])
        
        if scores[0][0] > 0.3:
            answer = self._format_answer(query, retrieved_chunks, sources)
        else:
            answer = "Не найдено релевантной информации для ответа на этот вопрос."
        
        return {
            "query": query,
            "answer": answer,
            "sources": sources,
            "retrieved_chunks": retrieved_chunks,
            "top_scores": [round(float(s), 4) for s in scores[0]]
        }
    
    def _format_answer(self, query: str, chunks: List[str], sources: List[dict]) -> str:
        answer_parts = []
        if chunks:
            answer_parts.append(f"{chunks[0][:300]}")            
        return "\n".join(answer_parts)

rag = MiniRAG(embedder, search_index, chunks_df)

In [24]:
test_queries = [
    "Сколько шагов насчитал Раскольников от своей каморки?",
    "Какое ранение получил Андрей Болконский?",
    "Как звали кота в Мастере и Маргарите?",
    "На каком языке Татьяна написала письмо Онегину?",
]

answers = []

for query in test_queries:
    
    result = rag.answer(query, top_k=3)
    
    answers.append({
        "question": query,
        "answer": result["answer"],
        "retrieved_sources": result["sources"]
    })

rag_examples = pd.DataFrame(answers)
rag_examples.to_csv("./artifacts/rag_examples.csv")
rag_examples

,question,answer,retrieved_sources
0,Сколько шагов насчитал Раскольников от своей каморки?,"Действие происходит в душной атмосфере Петербурга, где Раскольников мечется между желанием сознаться и страхом разоблачения. Большую роль в сюжете играет следователь Порфирий","[classic_01, new_03, classic_01]"
1,Какое ранение получил Андрей Болконский?,"чья первая любовь к князю Андрею Болконскому заканчивается трагически из-за его ранения при Бородине. Андрей Болконский — умный, скептичный и честолюбивый князь,","[classic_02, new_03, classic_09]"
2,Как звали кота в Мастере и Маргарите?,"и треснувшем пенсне, кот Бегемот, любящий водку, шахматы и трамвайные приключения, мрачный убийца Азазелло и прекрасная ведьма Гелла. Воланд и его свита","[classic_03, classic_03, classic_03]"
3,На каком языке Татьяна написала письмо Онегину?,"и семьёй Лариных. Татьяна Ларина, задумчивая и мечтательная провинциальная девушка, влюбляется в Онегина с первого взгляда и пишет ему знаменитое письмо на","[classic_04, classic_12, classic_04]"
